# Week 2.1: Getting Text Ready for Your Twin
### Plain-English edition — no coding required

Your digital twin only knows what you feed it. But real text is **messy**: extra spaces,
SHOUTING, typos, phone numbers buried in sentences, documents far too long to send at once.

This notebook is about **text surgery** — cutting, cleaning and slicing text until it is
in a shape your twin can actually use.

## How this notebook works
Every grey box below is a **step you run**, not code you read.

1. Some steps have **boxes to fill in** (type a word, drag a slider, tick a box).
2. Press the **▶ play button** on the left of the step.
3. Read the answer that appears underneath.

That's it. The computer writes and runs the code for you.

**Curious what it wrote?** Click **Show code** on any step to read it, then click
**Hide code** in the same place to tuck it away again.

> **Red text means something needs fixing** — read it and press play again.
> **Green means you got it right.** No red at all? You're good.

## What you will learn
- **Slicing** — grabbing an exact piece of a sentence by position
- **Cleaning** — turning messy input into tidy, consistent text
- **Searching** — finding out whether, where and how often something appears
- **Extracting** — pulling emails, times, numbers and links out of ordinary writing
- **Chunking** — chopping a long document into pieces small enough to send
- **The pipeline** — doing all of it in one move

---
## Part 1: A sentence is just a row of characters

To a computer your sentence is a numbered row of characters — character 0, character 1,
character 2 and so on. Once you know that, you can reach in and grab any piece you want.

In [ ]:
#@title ▶ Step 1 — Grab a piece of my sentence { display-mode: "form" }
#@markdown Type a sentence, then choose which slice of it to cut out.
my_sentence = "Hello! I love running at 6am every morning. Coffee is essential." #@param {type:"string"}
#@markdown Cut from this character position...
start_at = 7 #@param {type:"slider", min:0, max:80, step:1}
#@markdown ...up to (but not including) this one:
stop_at = 20 #@param {type:"slider", min:0, max:80, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number": r"\+?\d{1,3}[-. ]?\d{3}[-. ]?\d{3}[-. ]?\d{4}",
 "time of day": r"\d{1,2}:\d{2}\s?(?:am|pm)?",
 "date": r"\d{4}-\d{2}-\d{2}",
 "web link": r"https?://[^\s]+",
 "hashtag": r"#\w+",
 "@mention": r"@\w+",
 "any number": r"\b\d+(?:\.\d+)?\b",
 "duration": r"\d+\s?(?:h|hr|hours?|m|min|minutes?|km|k)\b",
}
PLAIN = {
 "email address": "something, then @, then something, then a dot and letters",
 "phone number": "an optional +, then groups of digits with dashes or spaces between them",
 "time of day": "one or two digits, a colon, two digits, maybe am or pm",
 "date": "four digits, dash, two digits, dash, two digits",
 "web link": "http:// or https:// followed by anything that is not a space",
 "hashtag": "a # followed by letters or numbers",
 "@mention": "an @ followed by letters or numbers",
 "any number": "one or more digits, maybe with a decimal point",
 "duration": "a number followed by a unit such as min, hr or km",
}
FILLER = {"the","a","an","and","or","but","is","are","was","were","in","on","at",
          "to","for","of","with","by","i","my","me","it","that","this"}
def show_text(label, text, width=76):
    print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

text = my_sentence
if len(text) < 3:
    problem("Please type a sentence first, then press play again.")
elif start_at >= stop_at:
    problem("'Cut from' (" + str(start_at) + ") must be SMALLER than 'up to' (" + str(stop_at) + ").")
    note("Drag the second slider further to the right, then press play again.")
else:
    show_text("Your sentence:", text)
    print("")
    print("It is " + str(len(text)) + " characters long, numbered 0 to " + str(len(text) - 1) + ".")
    print("")
    if start_at >= len(text):
        problem("There is no character at position " + str(start_at) + " - your sentence stops at "
                + str(len(text) - 1) + ".")
    else:
        piece = text[start_at:stop_at]
        print("Characters " + str(start_at) + " to " + str(stop_at - 1) + ":")
        print("")
        print("   '" + piece + "'")
        print("")
        marker = " " * min(start_at, 60) + "^" * max(1, len(piece[:60]))
        show_text("Where that sits:", text[:70])
        print("   " + marker[:70])
        print("")
        print("A few more cuts of the same sentence:")
        print("   the first character   -> '" + text[0] + "'")
        print("   the last character    -> '" + text[-1] + "'")
        print("   the first five        -> '" + text[:5] + "'")
        print("   the last ten          -> '" + text[-10:] + "'")
        print("   backwards             -> '" + text[::-1][:40] + ("..." if len(text) > 40 else "") + "'")
        print("")
        print("So what? Slicing is how you pull 'the bit after the colon' or 'the first")
        print("200 characters' out of a reply, without reading the whole thing yourself.")

---
## Part 2: Cleaning up messy input

People type like people: trailing spaces, CAPITALS, three exclamation marks. Two
sentences that mean the same thing can look completely different to a computer.
Cleaning makes them match.

**Our messy sample:** `"  I LOVE running, Coding AND   reading!!!  "`

In [ ]:
#@title ▶ Step 2 — Clean up a messy message { display-mode: "form" }
#@markdown Leave the messy sample, or paste in your own.
messy_text = "  I LOVE running, Coding AND   reading!!!  " #@param {type:"string"}
#@markdown Tick the cleaning steps you want, then press play:
trim_spaces = True #@param {type:"boolean"}
make_lowercase = True #@param {type:"boolean"}
remove_punctuation = True #@param {type:"boolean"}
drop_filler_words = False #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number": r"\+?\d{1,3}[-. ]?\d{3}[-. ]?\d{3}[-. ]?\d{4}",
 "time of day": r"\d{1,2}:\d{2}\s?(?:am|pm)?",
 "date": r"\d{4}-\d{2}-\d{2}",
 "web link": r"https?://[^\s]+",
 "hashtag": r"#\w+",
 "@mention": r"@\w+",
 "any number": r"\b\d+(?:\.\d+)?\b",
 "duration": r"\d+\s?(?:h|hr|hours?|m|min|minutes?|km|k)\b",
}
PLAIN = {
 "email address": "something, then @, then something, then a dot and letters",
 "phone number": "an optional +, then groups of digits with dashes or spaces between them",
 "time of day": "one or two digits, a colon, two digits, maybe am or pm",
 "date": "four digits, dash, two digits, dash, two digits",
 "web link": "http:// or https:// followed by anything that is not a space",
 "hashtag": "a # followed by letters or numbers",
 "@mention": "an @ followed by letters or numbers",
 "any number": "one or more digits, maybe with a decimal point",
 "duration": "a number followed by a unit such as min, hr or km",
}
FILLER = {"the","a","an","and","or","but","is","are","was","were","in","on","at",
          "to","for","of","with","by","i","my","me","it","that","this"}
def show_text(label, text, width=76):
    print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

if not messy_text.strip():
    problem("Please type some text first, then press play again.")
elif not (trim_spaces or make_lowercase or remove_punctuation or drop_filler_words):
    problem("Every box is unticked, so there is nothing to clean.")
    note("Tick at least one box, then press play again.")
else:
    print("Before:")
    print("   '" + messy_text + "'")
    print("   " + str(len(messy_text)) + " characters")
    print("")
    out = messy_text
    steps = []
    if trim_spaces:
        out = re.sub(r"\s+", " ", out).strip()
        steps.append("squeezed runs of spaces down to one and trimmed the ends")
    if make_lowercase:
        out = out.lower()
        steps.append("made everything lowercase")
    if remove_punctuation:
        out = re.sub(r"[^\w\s]", "", out)
        steps.append("removed punctuation")
    if drop_filler_words:
        kept = [w for w in out.split() if w.lower().strip(".,!?") not in FILLER]
        dropped = [w for w in out.split() if w.lower().strip(".,!?") in FILLER]
        out = " ".join(kept)
        steps.append("dropped filler words" + (" (" + ", ".join(dropped) + ")" if dropped else " (none found)"))
    print("What happened:")
    for s in steps:
        print("   - " + s)
    print("")
    print("After:")
    print("   '" + out + "'")
    print("   " + str(len(out)) + " characters, " + str(len(out.split())) + " words")
    print("")
    saved = len(messy_text) - len(out)
    if saved > 0:
        success("You cut " + str(saved) + " characters - that is " + str(saved)
                + " fewer characters to pay for on every single message.")
    print("")
    print("So what? 'I LOVE Running' and 'i love running' now look identical to the")
    print("computer. Without that step they would be treated as two different things.")

---
## Part 3: Finding things

Before you can act on text you often need to ask simple questions of it: *is this word
in there? where? how many times?*

In [ ]:
#@title ▶ Step 3 — Find a word in my text { display-mode: "form" }
search_in = "I love running at 6am. Running keeps me energized." #@param {type:"string"}
#@markdown What word should I look for?
look_for = "running" #@param {type:"string"}
#@markdown Should RUNNING, Running and running count as the same word?
ignore_capitals = True #@param {type:"boolean"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number": r"\+?\d{1,3}[-. ]?\d{3}[-. ]?\d{3}[-. ]?\d{4}",
 "time of day": r"\d{1,2}:\d{2}\s?(?:am|pm)?",
 "date": r"\d{4}-\d{2}-\d{2}",
 "web link": r"https?://[^\s]+",
 "hashtag": r"#\w+",
 "@mention": r"@\w+",
 "any number": r"\b\d+(?:\.\d+)?\b",
 "duration": r"\d+\s?(?:h|hr|hours?|m|min|minutes?|km|k)\b",
}
PLAIN = {
 "email address": "something, then @, then something, then a dot and letters",
 "phone number": "an optional +, then groups of digits with dashes or spaces between them",
 "time of day": "one or two digits, a colon, two digits, maybe am or pm",
 "date": "four digits, dash, two digits, dash, two digits",
 "web link": "http:// or https:// followed by anything that is not a space",
 "hashtag": "a # followed by letters or numbers",
 "@mention": "an @ followed by letters or numbers",
 "any number": "one or more digits, maybe with a decimal point",
 "duration": "a number followed by a unit such as min, hr or km",
}
FILLER = {"the","a","an","and","or","but","is","are","was","were","in","on","at",
          "to","for","of","with","by","i","my","me","it","that","this"}
def show_text(label, text, width=76):
    print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

word = look_for.strip()
if not search_in.strip():
    problem("Please type some text to search in, then press play again.")
elif not word:
    problem("Please type a word to look for, then press play again.")
else:
    hay = search_in.lower() if ignore_capitals else search_in
    needle = word.lower() if ignore_capitals else word
    show_text("Searching in:", search_in)
    print("")
    print("Looking for: '" + word + "'" + ("  (capitals ignored)" if ignore_capitals else "  (capitals matter)"))
    print("")
    count = hay.count(needle)
    if count == 0:
        problem("Not found: '" + word + "' does not appear in that text.")
        if not ignore_capitals and search_in.lower().count(needle.lower()) > 0:
            note("It IS there in different capitals. Tick 'ignore capitals' and press play again.")
    else:
        positions = []
        i = hay.find(needle)
        while i != -1:
            positions.append(i)
            i = hay.find(needle, i + 1)
        success("Found it " + str(count) + (" time" if count == 1 else " times") + ".")
        print("")
        print("At character position" + ("" if count == 1 else "s") + ": "
              + ", ".join(str(p) for p in positions))
        print("")
        print("Each one in context:")
        for p in positions:
            lo = max(0, p - 18); hi = min(len(search_in), p + len(word) + 18)
            print("   ..." + search_in[lo:p] + "[" + search_in[p:p + len(word)] + "]"
                  + search_in[p + len(word):hi] + "...")
        print("")
        if not ignore_capitals:
            other = search_in.lower().count(needle.lower()) - count
            if other > 0:
                note("There " + ("is " if other == 1 else "are ") + str(other) + " more in different capitals, which you are currently ignoring.")
    print("")
    print("So what? This is how a twin checks 'did they already tell me their name?'")
    print("before asking for it a second time.")

---
## Part 4: Pulling the useful bits out

Somewhere in an ordinary message there may be an email address, a time, a phone number.
A **pattern** describes the *shape* of the thing you want, so the computer can find every
one of them without you reading the text at all.

**Our sample message:**

```
Hi! I'm Sarah (sarah.runner@email.com). Reach me at +1-555-234-5678.
I run 5km daily at 6:30am and love #running #fitness.
My best time: 25min for 5km. Goal: 10km by 2026-03-01.
Follow me: https://strava.com/sarah_runs or @SarahRuns.
```

In [ ]:
#@title ▶ Step 4 — Pull out the useful bits { display-mode: "form" }
#@markdown Leave the sample message, or paste your own text.
message = "Hi! I'm Sarah (sarah.runner@email.com). Reach me at +1-555-234-5678. I run 5km daily at 6:30am and love #running #fitness. My best time: 25min for 5km. Goal: 10km by 2026-03-01. Follow me: https://strava.com/sarah_runs or @SarahRuns." #@param {type:"string"}
#@markdown What should I pull out?
look_for = "everything" #@param ["everything", "email address", "phone number", "time of day", "date", "web link", "hashtag", "@mention", "any number", "duration"]

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number": r"\+?\d{1,3}[-. ]?\d{3}[-. ]?\d{3}[-. ]?\d{4}",
 "time of day": r"\d{1,2}:\d{2}\s?(?:am|pm)?",
 "date": r"\d{4}-\d{2}-\d{2}",
 "web link": r"https?://[^\s]+",
 "hashtag": r"#\w+",
 "@mention": r"@\w+",
 "any number": r"\b\d+(?:\.\d+)?\b",
 "duration": r"\d+\s?(?:h|hr|hours?|m|min|minutes?|km|k)\b",
}
PLAIN = {
 "email address": "something, then @, then something, then a dot and letters",
 "phone number": "an optional +, then groups of digits with dashes or spaces between them",
 "time of day": "one or two digits, a colon, two digits, maybe am or pm",
 "date": "four digits, dash, two digits, dash, two digits",
 "web link": "http:// or https:// followed by anything that is not a space",
 "hashtag": "a # followed by letters or numbers",
 "@mention": "an @ followed by letters or numbers",
 "any number": "one or more digits, maybe with a decimal point",
 "duration": "a number followed by a unit such as min, hr or km",
}
FILLER = {"the","a","an","and","or","but","is","are","was","were","in","on","at",
          "to","for","of","with","by","i","my","me","it","that","this"}
def show_text(label, text, width=76):
    print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

if not message.strip():
    problem("Please paste some text first, then press play again.")
else:
    show_text("Your text:", message)
    print("")
    names = list(PATTERNS) if look_for == "everything" else [look_for]
    found_any = False
    for name in names:
        hits = re.findall(PATTERNS[name], message, re.IGNORECASE)
        hits = [h for h in hits if str(h).strip()]
        if hits:
            found_any = True
            seen = []
            for h in hits:
                if h not in seen: seen.append(h)
            print(name + ":")
            for h in seen:
                print("   " + str(h))
            print("   (the shape I looked for: " + PLAIN[name] + ")")
            print("")
    if not found_any:
        problem("I found no " + ("matches of any kind" if look_for == "everything" else look_for) + " in that text.")
        note("Try the sample message, or pick a different thing to look for.")
    else:
        success("Everything above was pulled out automatically - nobody read the text.")
    print("")
    print("So what? This is how a twin fills in a profile from a chat: it never")
    print("asks 'what is your email?', it just notices one going past.")

---
## Part 5: Chopping a long document into chunks

You cannot send a 50-page document to a model in one go. So you cut it into **chunks** —
pieces small enough to send, each one still making sense on its own.

**Our sample document** (3 paragraphs, 9 sentences, about 90 words):

```
A digital twin is a virtual representation of a physical object or system.
Digital twins are used across industries. In healthcare, they model patient health.
...
```

In [ ]:
#@title ▶ Step 5 — Chop a document into chunks { display-mode: "form" }
#@markdown How should I cut it up?
method = "by words, with overlap" #@param ["by sentences", "by words, with overlap", "by paragraph"]
#@markdown How big should each chunk be? (sentences, or words, depending on the method above)
chunk_size = 25 #@param {type:"slider", min:1, max:60, step:1}
#@markdown How much should neighbouring chunks overlap? (words - only used by the middle method)
overlap = 6 #@param {type:"slider", min:0, max:20, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number": r"\+?\d{1,3}[-. ]?\d{3}[-. ]?\d{3}[-. ]?\d{4}",
 "time of day": r"\d{1,2}:\d{2}\s?(?:am|pm)?",
 "date": r"\d{4}-\d{2}-\d{2}",
 "web link": r"https?://[^\s]+",
 "hashtag": r"#\w+",
 "@mention": r"@\w+",
 "any number": r"\b\d+(?:\.\d+)?\b",
 "duration": r"\d+\s?(?:h|hr|hours?|m|min|minutes?|km|k)\b",
}
PLAIN = {
 "email address": "something, then @, then something, then a dot and letters",
 "phone number": "an optional +, then groups of digits with dashes or spaces between them",
 "time of day": "one or two digits, a colon, two digits, maybe am or pm",
 "date": "four digits, dash, two digits, dash, two digits",
 "web link": "http:// or https:// followed by anything that is not a space",
 "hashtag": "a # followed by letters or numbers",
 "@mention": "an @ followed by letters or numbers",
 "any number": "one or more digits, maybe with a decimal point",
 "duration": "a number followed by a unit such as min, hr or km",
}
FILLER = {"the","a","an","and","or","but","is","are","was","were","in","on","at",
          "to","for","of","with","by","i","my","me","it","that","this"}
def show_text(label, text, width=76):
    print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

DOC = """A digital twin is a virtual representation of a physical object or system.
Digital twins are used across various industries. In healthcare, they model patient health.
In manufacturing, they simulate production processes. Smart cities use them for urban planning.

The technology relies on real-time data from sensors. This data feeds into simulation models.
Machine learning algorithms analyse the patterns. Predictions help optimise operations.

Benefits include better decisions and lower costs through predictive maintenance."""

print("The document: " + str(len(DOC.split())) + " words, "
      + str(len(sentences_of(DOC))) + " sentences, "
      + str(len([p for p in re.split(r"\n\s*\n", DOC) if p.strip()])) + " paragraphs.")
print("")
chunks = []
if method == "by sentences":
    sents = sentences_of(DOC)
    for i in range(0, len(sents), max(1, chunk_size)):
        chunks.append(" ".join(sents[i:i + max(1, chunk_size)]))
    note("Cutting every " + str(chunk_size) + " sentence(s). The 'overlap' slider is ignored by this method.")
elif method == "by paragraph":
    chunks = [p.strip().replace("\n", " ") for p in re.split(r"\n\s*\n", DOC) if p.strip()]
    note("Cutting at blank lines. The size and overlap sliders are ignored by this method.")
else:
    words = DOC.split()
    if overlap >= chunk_size:
        problem("Overlap (" + str(overlap) + ") must be SMALLER than chunk size (" + str(chunk_size) + ").")
        note("Otherwise each chunk would repeat the one before it forever. Fix a slider and press play again.")
        words = []
    step = max(1, chunk_size - overlap)
    for i in range(0, len(words), step):
        piece = words[i:i + chunk_size]
        if piece: chunks.append(" ".join(piece))
        if i + chunk_size >= len(words): break
print("")
if chunks:
    print("Result: " + str(len(chunks)) + " chunk(s).")
    print("")
    for i, c in enumerate(chunks, 1):
        print("Chunk " + str(i) + "  (" + str(len(c.split())) + " words)")
        show_text("", c)
        print("")
    sizes = [len(c.split()) for c in chunks]
    print("Smallest chunk: " + str(min(sizes)) + " words | biggest: " + str(max(sizes)) + " words")
    if method == "by words, with overlap" and overlap > 0 and len(chunks) > 1:
        shared = " ".join(chunks[0].split()[-overlap:])
        success("Chunks 1 and 2 share these " + str(overlap) + " words: \"" + shared + "\"")
        print("")
        print("That repetition is deliberate - the next step shows why.")

---
### Why overlap is worth paying for

Cutting blindly can slice a fact in half. The chunk that holds *"my best 5km time"*
ends before the number, and the chunk with the number no longer says what it is a time for.
Overlap repeats a few words at each join so facts survive the cut.

In [ ]:
#@title ▶ Step 6 — Watch a fact get cut in half { display-mode: "form" }
#@markdown A short note about you. The fact we care about is "best 5km time is 25 minutes".
my_note = "My name is Sarah. I run every morning before work. My best 5km time is 25 minutes. I am training for a 10km race." #@param {type:"string"}
#@markdown Cut into chunks of this many words:
chunk_size = 7 #@param {type:"slider", min:4, max:20, step:1}
#@markdown Repeat this many words at each join:
overlap = 0 #@param {type:"slider", min:0, max:10, step:1}
#@markdown Which fact are we trying to keep in one piece?
the_fact = "5km time is 25 minutes" #@param {type:"string"}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number": r"\+?\d{1,3}[-. ]?\d{3}[-. ]?\d{3}[-. ]?\d{4}",
 "time of day": r"\d{1,2}:\d{2}\s?(?:am|pm)?",
 "date": r"\d{4}-\d{2}-\d{2}",
 "web link": r"https?://[^\s]+",
 "hashtag": r"#\w+",
 "@mention": r"@\w+",
 "any number": r"\b\d+(?:\.\d+)?\b",
 "duration": r"\d+\s?(?:h|hr|hours?|m|min|minutes?|km|k)\b",
}
PLAIN = {
 "email address": "something, then @, then something, then a dot and letters",
 "phone number": "an optional +, then groups of digits with dashes or spaces between them",
 "time of day": "one or two digits, a colon, two digits, maybe am or pm",
 "date": "four digits, dash, two digits, dash, two digits",
 "web link": "http:// or https:// followed by anything that is not a space",
 "hashtag": "a # followed by letters or numbers",
 "@mention": "an @ followed by letters or numbers",
 "any number": "one or more digits, maybe with a decimal point",
 "duration": "a number followed by a unit such as min, hr or km",
}
FILLER = {"the","a","an","and","or","but","is","are","was","were","in","on","at",
          "to","for","of","with","by","i","my","me","it","that","this"}
def show_text(label, text, width=76):
    print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

words = my_note.split()
if len(words) < 6:
    problem("Please write a slightly longer note, then press play again.")
elif overlap >= chunk_size:
    problem("Overlap (" + str(overlap) + ") must be smaller than chunk size (" + str(chunk_size) + ").")
else:
    step = max(1, chunk_size - overlap)
    chunks = []
    for i in range(0, len(words), step):
        piece = words[i:i + chunk_size]
        if piece: chunks.append(" ".join(piece))
        if i + chunk_size >= len(words): break
    print("Cut into " + str(len(chunks)) + " chunks of up to " + str(chunk_size)
          + " words, repeating " + str(overlap) + " at each join:")
    print("")
    target = " ".join(the_fact.lower().split())
    survivor = None
    for i, c in enumerate(chunks, 1):
        flat = " ".join(c.lower().split())
        holds = target in flat
        if holds and survivor is None: survivor = i
        print("Chunk " + str(i) + ("   <-- holds the whole fact" if holds else ""))
        show_text("", c)
    print("")
    if survivor:
        success("The fact \"" + the_fact + "\" survived intact, in chunk " + str(survivor) + ".")
        print("")
        print("A search for 'how fast does she run 5km?' can now find it.")
    else:
        problem("The fact \"" + the_fact + "\" did not survive - it is split across chunks.")
        print("")
        print("No single chunk answers 'how fast does she run 5km?' any more.")
        note("Now drag the overlap slider up a few notches and press play again.")

---
## Part 6: The whole pipeline, in one move

In a real twin these steps do not happen one at a time by hand. Raw text goes in one end,
and clean, labelled, chunked text comes out the other.

In [ ]:
#@title ▶ Step 7 — Run the whole pipeline { display-mode: "form" }
#@markdown Paste anything — a message, a note, part of a document.
raw_text = "Hi!! I'm Alex (alex@mail.com), I  wake at 6:30am and run 5km most days. Training log: https://log.me/alex  #running" #@param {type:"string"}
clean_it = True #@param {type:"boolean"}
extract_the_details = True #@param {type:"boolean"}
chop_into_chunks = True #@param {type:"boolean"}
#@markdown Words per chunk (if chopping):
chunk_size = 12 #@param {type:"slider", min:4, max:60, step:1}

# ---- hidden helper code: learners never need to read this ----
import sys, numpy as np, matplotlib.pyplot as plt
from collections import Counter
from IPython.display import display, HTML
def _msg(text, colour, weight):
    sys.stdout.flush()
    body = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    display(HTML('<pre style="margin:0 0 2px 0;font-family:monospace;white-space:pre-wrap;'
                 'font-size:13px;color:' + colour + ';font-weight:' + weight + '">' + body + '</pre>'))
def problem(text): _msg(text, "#C8283C", "700")
def success(text): _msg(text, "#1a7f37", "700")
def note(text):    _msg(text, "#5f6368", "400")
def words_of(text):
    import re
    return [w for w in re.findall(r"[a-zA-Z']+", text.lower()) if w]

import re
PATTERNS = {
 "email address": r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
 "phone number": r"\+?\d{1,3}[-. ]?\d{3}[-. ]?\d{3}[-. ]?\d{4}",
 "time of day": r"\d{1,2}:\d{2}\s?(?:am|pm)?",
 "date": r"\d{4}-\d{2}-\d{2}",
 "web link": r"https?://[^\s]+",
 "hashtag": r"#\w+",
 "@mention": r"@\w+",
 "any number": r"\b\d+(?:\.\d+)?\b",
 "duration": r"\d+\s?(?:h|hr|hours?|m|min|minutes?|km|k)\b",
}
PLAIN = {
 "email address": "something, then @, then something, then a dot and letters",
 "phone number": "an optional +, then groups of digits with dashes or spaces between them",
 "time of day": "one or two digits, a colon, two digits, maybe am or pm",
 "date": "four digits, dash, two digits, dash, two digits",
 "web link": "http:// or https:// followed by anything that is not a space",
 "hashtag": "a # followed by letters or numbers",
 "@mention": "an @ followed by letters or numbers",
 "any number": "one or more digits, maybe with a decimal point",
 "duration": "a number followed by a unit such as min, hr or km",
}
FILLER = {"the","a","an","and","or","but","is","are","was","were","in","on","at",
          "to","for","of","with","by","i","my","me","it","that","this"}
def show_text(label, text, width=76):
    print(label)
    for line in text.splitlines():
        while len(line) > width:
            cut = line.rfind(" ", 0, width)
            if cut <= 0: cut = width
            print("   " + line[:cut]); line = line[cut:].lstrip()
        print("   " + line)
def sentences_of(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]

if not raw_text.strip():
    problem("Please paste some text first, then press play again.")
elif not (clean_it or extract_the_details or chop_into_chunks):
    problem("Every stage is switched off, so the pipeline would do nothing.")
    note("Tick at least one box, then press play again.")
else:
    show_text("IN:", raw_text)
    print("   " + str(len(raw_text)) + " characters, " + str(len(raw_text.split())) + " words")
    print("")
    work = raw_text
    done = []
    if clean_it:
        work = re.sub(r"\s+", " ", work).strip()
        done.append("cleaned")
        print("-> cleaned")
        show_text("", work)
        print("")
    if extract_the_details:
        done.append("details extracted")
        print("-> details found")
        any_hit = False
        for name, pat in PATTERNS.items():
            hits = re.findall(pat, work, re.IGNORECASE)
            if hits:
                any_hit = True
                seen = []
                for h in hits:
                    if h not in seen: seen.append(h)
                print("   " + name + ": " + ", ".join(str(h) for h in seen))
        if not any_hit:
            print("   (none found in this text)")
        print("")
    chunks = []
    if chop_into_chunks:
        w = work.split()
        for i in range(0, len(w), max(1, chunk_size)):
            chunks.append(" ".join(w[i:i + max(1, chunk_size)]))
        done.append(str(len(chunks)) + " chunks")
        print("-> chopped into " + str(len(chunks)) + " chunk(s)")
        for i, c in enumerate(chunks, 1):
            print("   [" + str(i) + "] " + c)
        print("")
    print("OUT: " + str(len(work)) + " characters, " + str(len(work.split())) + " words"
          + (", " + str(len(chunks)) + " chunks ready to store" if chunks else ""))
    print("")
    success("Pipeline finished: " + " -> ".join(done) + ".")
    print("")
    print("Every piece above is now in a shape your twin can store, search and send.")

---
## What you just did

| Step | In plain words | Why your twin needs it |
|---|---|---|
| 1 | Grabbed a slice by position | Pull one field out of a reply |
| 2 | Cleaned messy input | So "I LOVE" and "i love" count as the same thing |
| 3 | Searched the text | Check what you already know before asking again |
| 4 | Extracted emails, times, numbers | Fill in a profile without asking questions |
| 5 | Chopped a document into chunks | Long documents will not fit in one message |
| 6 | Added overlap at the joins | So facts do not get cut in half |
| 7 | Ran the whole pipeline | This is what really happens before text is stored |

### Where this is going
Chunks are useless on their own — you need to find *the right chunk* for a question.
That needs each chunk turned into numbers (**Week 2.3: Embeddings**) and searched
(**Week 2.4: RAG**). Next up, though: why a document has to be chopped at all —
**Week 2.2: Context Window Management**.

### Want to see the code?
Click **Show code** on any step. Everything above was written for you.